In [1]:
import pandas as pd
import numpy as np
import glob

# Load and inspect NFI data tables

In [4]:
stub = pd.read_csv('stub.csv')
source = pd.read_csv('source.csv')
stub_source = pd.read_csv('stub_source.csv')
particle = pd.concat([pd.read_csv(f) for f in sorted(glob.glob('particle_*.csv'))], ignore_index=True)

In [10]:
print("Particle classes in target field") 
print(particle['merged_relevance_class'].value_counts())

print("\nRelevance classes (un-merged)")
print(particle['relevance_class'].value_counts())

Particle classes in target field
merged_relevance_class
BaCaSi      32816
PbSbBa      30299
CuZn        29975
BaAl        22832
PbBa        20570
Sb          18928
Pb          16309
PbSb        13382
BaSb         5726
Ba           5315
ZnTi         3313
PbSbBaSn     2313
Sr           1734
TiZnGd       1447
SbHg          225
GaCuSn        159
BaSn           91
Hg             70
PbSbBaSr        3
Name: count, dtype: int64

Relevance classes (un-merged)
relevance_class
PbSbBa      30299
CuZn        29975
BaCaSiS     22721
PbBa        20270
BaAlS       17586
Pb          16309
Sb          15056
PbSb        11955
BaCaSi      10095
BaAl         5246
BaSb         4897
SbSn         3872
BaSi         3783
ZnTi         3313
PbSbBaSn     2313
Sr           1734
TiZnGd       1447
PbSbSn       1427
BaSbSn        829
BaSr          822
Ba            710
PbBaSn        300
SbHg          225
GaCuSn        159
BaSn           91
Hg             70
PbSbBaSr        3
Name: count, dtype: int64


In [11]:
merge_map = {
    'PbSbBa': 'PbBaSb',
    'PbSbBaSn': 'PbBaSb',
    'PbSbBaSr': 'PbBaSb',
    'PbBa': 'PbBa',
    'PbBaSn': 'PbBa',
    'PbSb': 'PbSb',
    'PbSbSn': 'PbSb',
    'BaSb': 'BaSb',
    'BaSbSn': 'BaSb',
    'Sb': 'Sb',
    'SbSn': 'Sb',
    'Ba': 'Ba',
    'BaSi': 'Ba',
    'BaSr': 'Ba',
    'BaSn': 'Ba',
    'BaAl': 'BaAl',
    'BaAlS': 'BaAl',
    'BaCaSi': 'BaCaSi',
    'BaCaSiS': 'BaCaSi',
    'CuZn': 'CuZn',
    'Pb': 'Pb',
    'ZnTi': 'ZnTi',
    'Sr': 'Sr',
    'Hg': 'Hg',
    'SbHg': 'Hg',
    'TiZnGd': 'TiZnGd',
    'GaCuSn': 'GaCuSn',
}

particle['final_class'] = particle['relevance_class'].map(merge_map)

/var/folders/1f/1zydwygj5qnbp9nc3jfjbdwc0000gr/T/ipykernel_98693/891891520.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  particle['final_class'] = particle['relevance_class'].map(merge_map)


In [12]:
unmapped = particle[particle['final_class'].isna()]['relevance_class'].unique()
print(f"Unmapped classes: {unmapped}")

print("\nFully merged class distribution:")
print(particle['final_class'].value_counts())

Unmapped classes: [nan]

Fully merged class distribution:
final_class
BaCaSi    32816
PbBaSb    32615
CuZn      29975
BaAl      22832
PbBa      20570
Sb        18928
Pb        16309
PbSb      13382
BaSb       5726
Ba         5406
ZnTi       3313
Sr         1734
TiZnGd     1447
Hg          295
GaCuSn      159
Name: count, dtype: int64


In [5]:
rd_stubs = stub[stub['type_project'] == 'R&D']
print("Distinct R&D projects:", rd_stubs['project'].nunique())
print("Distinct R&D sample_sources:", rd_stubs['sample_source'].nunique())
print("Distinct R&D sample_types:", rd_stubs['sample_type'].nunique())

print("\nSample type distribution:")
print(rd_stubs['sample_type'].value_counts())
print("\nSample source distribution:")
print(rd_stubs['sample_source'].value_counts())

Distinct R&D projects: 63
Distinct R&D sample_sources: 5
Distinct R&D sample_types: 6

Sample type distribution:
sample_type
hand                150
positive control    106
clothing             74
cartridge            68
inshot               35
blanco               10
Name: count, dtype: int64

Sample source distribution:
sample_source
suspect             168
positive control    106
unknown              74
victim               35
blanco               10
Name: count, dtype: int64


# Merging 

In [17]:
particle_stub = particle.merge(stub, left_on='stub_id', right_on='id',suffixes=('_particle', '_stub'))

In [18]:
rd_particles = particle_stub[particle_stub['type_project'] == 'R&D']

In [19]:
source_profiles = rd_particles.groupby('sample_source').agg(
    total_particles=('particle_id', 'count'),
    characteristic_count=('final_class', lambda x: (x == 'PbBaSb').sum()),
    unique_classes=('final_class', 'nunique')
).reset_index()

In [20]:
source_profiles['characteristic_proportion'] = (source_profiles['characteristic_count'] / source_profiles['total_particles'])
print(source_profiles.sort_values('characteristic_proportion', ascending=False).to_string())

      sample_source  total_particles  characteristic_count  unique_classes  characteristic_proportion
2           unknown              755                   130              13                   0.172185
3            victim             8812                  1256               9                   0.142533
1           suspect             9036                   791              15                   0.087539
0  positive control               26                     0               2                   0.000000


# Particle class by Source + Sample

In [21]:
rd_profile = pd.crosstab(
    rd_particles['sample_source'],
    rd_particles['final_class'],
    normalize='index'
).round(3)

print(rd_profile.to_string())

final_class          Ba   BaAl  BaCaSi   BaSb   CuZn  GaCuSn     Hg     Pb   PbBa  PbBaSb   PbSb     Sb     Sr  TiZnGd   ZnTi
sample_source                                                                                                                
positive control  0.000  0.000   0.000  0.000  0.000   0.000  0.000  0.000  0.615   0.000  0.385  0.000  0.000   0.000  0.000
suspect           0.005  0.052   0.016  0.028  0.397   0.018  0.001  0.084  0.032   0.088  0.122  0.037  0.003   0.108  0.011
unknown           0.005  0.107   0.040  0.101  0.155   0.000  0.003  0.181  0.028   0.172  0.148  0.037  0.000   0.019  0.004
victim            0.002  0.014   0.000  0.023  0.000   0.000  0.000  0.247  0.212   0.143  0.348  0.010  0.000   0.000  0.000


In [22]:
type_profile = pd.crosstab(
    rd_particles['sample_type'],
    rd_particles['final_class'],
    normalize='index'
).round(3)

print(type_profile.to_string())

final_class          Ba   BaAl  BaCaSi   BaSb   CuZn  GaCuSn     Hg     Pb   PbBa  PbBaSb   PbSb     Sb     Sr  TiZnGd   ZnTi
sample_type                                                                                                                  
cartridge         0.019  0.004   0.002  0.079  0.367   0.000  0.000  0.004  0.002   0.477  0.007  0.010  0.002   0.023  0.003
clothing          0.005  0.107   0.040  0.101  0.155   0.000  0.003  0.181  0.028   0.172  0.148  0.037  0.000   0.019  0.004
hand              0.006  0.067   0.020  0.037  0.210   0.024  0.002  0.113  0.044   0.119  0.165  0.050  0.003   0.128  0.013
inshot            0.002  0.014   0.000  0.023  0.000   0.000  0.000  0.247  0.212   0.143  0.348  0.010  0.000   0.000  0.000
positive control  0.000  0.000   0.000  0.000  0.000   0.000  0.000  0.000  0.615   0.000  0.385  0.000  0.000   0.000  0.000


# Case data

In [23]:
case_particles = particle_stub[particle_stub['type_project'] == 'zaak']

print(f"n Case particles: {len(case_particles)}")
print(f"n R&D particles:  {len(rd_particles)}")

print("\nCase class distribution:")
print(case_particles['final_class'].value_counts())

print("\nCase sample_source distribution:")
print(case_particles['sample_source'].value_counts())

print("\nCase sample_type distribution:")
print(case_particles['sample_type'].value_counts())

n Case particles: 183057
n R&D particles:  22450

Case class distribution:
final_class
BaCaSi    32644
PbBaSb    27282
CuZn      26271
BaAl      22159
Sb        18422
PbBa      18362
Pb        13234
PbSb       9077
Ba         5223
BaSb       4731
ZnTi       3207
Sr         1703
TiZnGd      461
Hg          281
Name: count, dtype: int64

Case sample_source distribution:
sample_source
suspect             79256
unknown             11085
positive control      120
blanco                 53
Name: count, dtype: int64

Case sample_type distribution:
sample_type
hand                55126
clothing            31047
cartridge           18730
car                  3553
body                  283
positive control      120
blanco                 53
Name: count, dtype: int64
